<a href="https://colab.research.google.com/github/deepthivj-aiml/Detect_sleep_states_OPTIMIZED_FOR_GPU_Kaggle/blob/main/Copy_of_detect_sleep_states_OPTIMIZED_FOR_GPU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


Kaggle credentials set.
Kaggle credentials successfully validated.


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

child_mind_institute_detect_sleep_states_path = kagglehub.competition_download('child-mind-institute-detect-sleep-states')

print('Data source import complete.')


Data source import complete.


# Import Data

In [ ]:
# data science
import numpy as np
import polars as pl
import pandas as pd
import datetime as dt

# machine learning
import xgboost as xgb_lib          # low-level API for incremental training + DMatrix
from tqdm import tqdm
import gc

# reproducibility
SEED = 9
np.random.seed(SEED)

In [ ]:
# read in the data
root = child_mind_institute_detect_sleep_states_path
data = pl.read_parquet(root + '/train_series.parquet')
events = pl.read_csv(root + '/train_events.csv')
test = pl.read_parquet(root + '/test_series.parquet')

# cast step
events = events.with_columns([pl.col("step").cast(pl.UInt32).alias("step")])

# downcast float64 sensor columns to float32 to cut memory ~50% for the rest of the pipeline
float_cols = [c for c, dt_ in zip(data.columns, data.dtypes) if dt_ == pl.Float64]
data = data.with_columns([pl.col(c).cast(pl.Float32) for c in float_cols])
test = test.with_columns([pl.col(c).cast(pl.Float32) for c in float_cols if c in test.columns])

# Clean Events

In [ ]:
# create (onsent, wakeup) pairs
pairs = (events.pivot(values="step", index=["series_id", "night"], on="event"))

# filter the pairs for rows where exactly one of the two is null
faulty_pairs = pairs.filter((pl.col("onset").is_null() & pl.col("wakeup").is_not_null()) | (pl.col("onset").is_not_null() & pl.col("wakeup").is_null()))

# view
faulty_pairs

series_id,night,onset,wakeup
str,i64,u32,u32
"""0ce74d6d2106""",20,332376,null
"""154fe824ed87""",30,null,514980
"""44a41bba1ee7""",10,null,165684
"""efbfc4526d58""",7,null,114864
"""f8a8da8bdd00""",17,null,291384


In [ ]:
# iterate through faulty_pairs['series_id', 'night'] and index into events
for series_id, night in zip(faulty_pairs['series_id'], faulty_pairs['night']):
    # fill all "step" and "timestamp" values with null
    events = events.with_columns(
        pl.when((pl.col('series_id') == series_id) & (pl.col('night') == night)).then(None).otherwise(pl.col('step')).alias('step'),
        pl.when((pl.col('series_id') == series_id) & (pl.col('night') == night)).then(None).otherwise(pl.col('timestamp')).alias('timestamp'))

# check again for mismatches
pairs = (events.pivot(values="step", index=["series_id", "night"], on="event"))
faulty_pairs = pairs.filter((pl.col("onset").is_null() & pl.col("wakeup").is_not_null()) | (pl.col("onset").is_not_null() & pl.col("wakeup").is_null()))
faulty_pairs

series_id,night,onset,wakeup
str,i64,u32,u32


# Timestamp

In [ ]:
def add_date_cols(df):
    """
    Add date columns to a DataFrame.

    Args:
    - df (pl.DataFrame): The DataFrame to modify.

    Returns:
    - df (pl.DataFrame): The modified DataFrame.
    """

    df = df.with_columns([
        # remove the timezone offset (get first 19 characters), convert to datetime
        pl.col('timestamp').str.slice(0, 19).str.strptime(pl.Datetime, format="%Y-%m-%dT%H:%M:%S").alias('timestamp_datetime')]).with_columns([

            # create date and hour columns
            pl.col('timestamp_datetime').dt.date().alias('date'),
            pl.col('timestamp_datetime').dt.hour().alias('hour')])

    # drop the unnecessary columns
    df = df.drop(['timestamp', 'timestamp_datetime'])

    return df

In [ ]:
# add date cols
data = add_date_cols(data)
events = add_date_cols(events)
test = add_date_cols(test)

# check range of dates
events.filter(pl.col('date').is_not_null())['date'].min(), events.filter(pl.col('date').is_not_null())['date'].max()

(datetime.date(2017, 8, 5), datetime.date(2019, 7, 5))

- Dates range from August 5th, 2017 - July 5th, 2019.

# Create Label

In [ ]:
def label_data(df):
    """
    Create an "asleep" label column (1 for asleep, 0 for awake) based on the "event" column.

    Args:
    - df (pl.DataFrame): DataFrame containing sleep data.

    Returns:
    - (pl.DataFrame): DataFrame with the "asleep" label column added.
    """

    df = df.sort(["series_id", "step"]).with_columns(
        pl.when(pl.col("event") == "onset")
        .then(1)
        .when(pl.col("event") == "wakeup")
        .then(-1)
        .otherwise(0)
        .alias("sleep_indicator"))

    df = df.with_columns(pl.col("sleep_indicator").cum_sum().over("series_id").alias("sleep_cumsum"))

    # store as Int8 instead of bool->later-upcast, saves memory downstream
    df = df.with_columns((pl.col("sleep_cumsum") > 0).cast(pl.Int8).alias("asleep"))

    return df.drop(['event', 'sleep_indicator', 'sleep_cumsum'])

In [ ]:
# label data with events
data = data.join(events[['series_id', 'step', 'event']], on=['series_id', 'step'], how='left').sort(by=['series_id', 'step'])

# add "asleep" column
data_labeled = label_data(data)

# check
data_labeled.sample()

series_id,step,anglez,enmo,date,hour,asleep
str,u32,f32,f32,date,i8,i8
"""7df249527c63""",297489,-31.465,0.0342,2017-11-23,20,0


# Train Model

In [ ]:
def create_features(df):
    """
    Create features for each series.

    Args:
    - df: contains the series data with columns 'enmo', 'anglez'

    Returns:
    - pl.dataframe: dataframe with new features added.
    """

    df = df.sort(["series_id", "step"])
    lazy_df = df.lazy()

    lazy_df = lazy_df.with_columns([
        (pl.col('anglez').diff().abs().fill_null(0)).alias('anglez_diff'),
        (pl.col('enmo').diff().abs().fill_null(0)).alias('enmo_diff')
    ])

    # guard both fill_nan AND fill_null since edge windows can produce nulls, not just NaNs
    agg_funcs = {
        "min":  lambda col, w: pl.col(col).rolling_min(w, center=True).fill_nan(0).fill_null(0),
        "max":  lambda col, w: pl.col(col).rolling_max(w, center=True).fill_nan(0).fill_null(0),
        "mean": lambda col, w: pl.col(col).rolling_mean(w, center=True).fill_nan(0).fill_null(0),
        "std":  lambda col, w: pl.col(col).rolling_std(w, center=True).fill_nan(0).fill_null(0),
    }

    # NOTE: trimmed from the original 15 windows to fewer, less-redundant ones to cut
    # feature count (and memory/train time) substantially. Restore the full list if you
    # want to test whether the extra windows help your validation score.
    windows = [1, 5, 15, 30, 60, 120, 240, 480]

    for m in windows:
        window_size = int(m * 12)
        exprs = []

        for col in ['anglez', 'enmo']:
            for stat, func in agg_funcs.items():
                exprs.append(func(col, window_size).alias(f'{col}_{m}m_{stat}'))

            diff_col = f'{col}_diff'
            for stat, func in agg_funcs.items():
                exprs.append(func(diff_col, window_size).alias(f'{diff_col}_{m}m_{stat}'))

        lazy_df = lazy_df.with_columns(exprs)

    return lazy_df.collect()

In [ ]:
def batch_data(data, batch_size=1_000_000, shuffle_series=True, seed=SEED):
    """
    Create batches of data for training.

    Args:
    - data (pl.DataFrame): Data to be batched.
    - batch_size (int): Size of each batch. Default is 1 million.
    - shuffle_series (bool): Randomize series order so each batch mixes people
      instead of being dominated by whichever series sort first.
    - seed (int): RNG seed for shuffling.

    Returns:
    - (generator): Yields feature-engineered batches.
    """

    if shuffle_series:
        series_ids = data.select("series_id").unique().to_series().to_list()
        rng = np.random.default_rng(seed)
        rng.shuffle(series_ids)
        order_map = {sid: i for i, sid in enumerate(series_ids)}
        data = data.with_columns(
            pl.col("series_id").replace(order_map).alias("_shuffle_key")
        ).sort(["_shuffle_key", "step"]).drop("_shuffle_key")

    for i in range(0, len(data), batch_size):
        batch = data[i:i + batch_size]
        yield create_features(batch)


def n_batches_for(data, batch_size=1_000_000):
    return (len(data) + batch_size - 1) // batch_size

In [ ]:
# ---- training config ----
non_feat_cols = ['series_id', 'step', 'date', 'asleep']
batch_size = 1_000_000
rounds_per_batch = 50

# class imbalance: weight the minority class
label_counts = data_labeled.select(pl.col("asleep").value_counts()).unnest("asleep")
counts = dict(zip(label_counts["asleep"], label_counts["count"]))
scale_pos_weight = counts.get(0, 1) / max(counts.get(1, 1), 1)

params = {
    'tree_method': 'hist',
    'objective': 'binary:logistic',
    'eval_metric': 'logloss',
    'max_depth': 6,
    'eta': 0.1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'scale_pos_weight': scale_pos_weight,
    'seed': SEED,
    'nthread': -1,
    # 'device': 'cuda',  # uncomment if a GPU is available
}

n_batches = n_batches_for(data_labeled, batch_size)
booster = None

for i, features in tqdm(enumerate(batch_data(data_labeled, batch_size)), total=n_batches, desc='Iterating through batches'):
    X_batch = features.drop(non_feat_cols).to_numpy().astype('float32')
    y_batch = features.select("asleep").to_numpy().ravel().astype('int32')

    # simple in-batch train/valid split for early stopping monitoring
    n = len(y_batch)
    cut = int(n * 0.9)
    dtrain = xgb_lib.DMatrix(X_batch[:cut], label=y_batch[:cut])
    dvalid = xgb_lib.DMatrix(X_batch[cut:], label=y_batch[cut:])

    booster = xgb_lib.train(
        params,
        dtrain,
        num_boost_round=rounds_per_batch,
        xgb_model=booster,
        evals=[(dvalid, 'valid')],
        early_stopping_rounds=10,
        verbose_eval=False,
    )

    del X_batch, y_batch, dtrain, dvalid, features
    gc.collect()

xgb = booster  # keep the 'xgb' name so later cells still work

Iterating through batches: 100%|██████████| 128/128 [24:48<00:00, 11.63s/it]


# Test Set Predictions

In [ ]:
def predict(data, classifier):
    """
    Takes a time series of (containing features and labels) and a classifier and returns a formatted submission dataframe.

    Args:
    - data (pl.DataFrame): Contains series data and sleep events.
    - classifier (xgb_lib.Booster): Trained booster for prediction.

    Returns:
    - event_preds_df (pd.DataFrame): Contains predicted sleep events.
    """

    # Dynamically adjust columns to drop for prediction
    cols_to_drop = ['series_id', 'step', 'date']
    if 'asleep' in data.columns: # Check if 'asleep' exists in the input data
        cols_to_drop.append('asleep')

    series_ids = data.select("series_id").unique().to_series().to_list()
    event_preds = []

    for sid in tqdm(series_ids, desc="Processing users"):
        user_data = data.filter(pl.col("series_id") == sid).sort("step")
        features = create_features(user_data)

        X = features.drop(cols_to_drop)
        X_np = X.to_numpy().astype('float32')
        dmat = xgb_lib.DMatrix(X_np)

        probs = classifier.predict(dmat)          # Booster.predict returns probabilities directly
        preds = (probs >= 0.5).astype(int)

        X = X.with_columns([
            pl.Series("step", features["step"]),
            pl.Series("date", features["date"]),
            pl.Series("pred", preds),
            pl.Series("prob", probs)
        ])

        X = X.with_columns(pl.col("pred").diff().alias("pred_diff"))

        pred_onsets = X.filter(pl.col("pred_diff") > 0)["step"].to_list()
        pred_wakeups = X.filter(pl.col("pred_diff") < 0)["step"].to_list()

        if len(pred_onsets) > 0 and len(pred_wakeups) > 0:
            if pred_wakeups[0] < pred_onsets[0]:
                pred_wakeups = pred_wakeups[1:]
            if pred_onsets and pred_wakeups and pred_onsets[-1] > pred_wakeups[-1]:
                pred_onsets = pred_onsets[:-1]

            segments = [(onset, wakeup) for onset, wakeup in zip(pred_onsets, pred_wakeups) if (wakeup - onset) >= (30 * 12)]

            if segments:
                merged_segments = []
                current_start, current_end = segments[0]
                for onset, wakeup in segments[1:]:
                    if onset - current_end < (120 * 12):
                        current_end = wakeup
                    else:
                        merged_segments.append((current_start, current_end))
                        current_start, current_end = onset, wakeup
                merged_segments.append((current_start, current_end))

                segments_by_night = {}
                for onset, wakeup in merged_segments:
                    night_key = X.filter(pl.col("step") == onset).select(pl.col("date")).to_series()[0]
                    duration = wakeup - onset
                    if night_key not in segments_by_night or duration > segments_by_night[night_key]["duration"]:
                        segments_by_night[night_key] = {"onset": onset, "wakeup": wakeup, "duration": duration}

                for night_key, seg in segments_by_night.items():
                    onset_step, wakeup_step = seg["onset"], seg["wakeup"]
                    sleep_segment = X.filter((pl.col("step") >= onset_step) & (pl.col("step") < wakeup_step))
                    score = sleep_segment.select(pl.col("prob")).mean().item()

                    onset_date = X.filter(pl.col("step") == onset_step).select(pl.col("date")).to_series()[0]
                    wakeup_date = X.filter(pl.col("step") == wakeup_step).select(pl.col("date")).to_series()[0]

                    event_preds.append({"series_id": sid, "step": onset_step, "event": "onset", "score": score, "date": onset_date})
                    event_preds.append({"series_id": sid, "step": wakeup_step, "event": "wakeup", "score": score, "date": wakeup_date})

    if event_preds:
        event_preds_df = pd.DataFrame(event_preds)
    else:
        event_preds_df = pd.DataFrame(columns=['series_id', 'step', 'event', 'score', 'date'])

    event_preds_df['row_id'] = range(len(event_preds_df))
    return event_preds_df[['row_id', 'series_id', 'step', 'event', 'score', 'date']]

In [ ]:
test

series_id,step,anglez,enmo,date,hour
str,u32,f32,f32,date,i8
"""038441c925bb""",0,2.6367,0.0217,2018-08-14,15
"""038441c925bb""",1,2.6368,0.0215,2018-08-14,15
"""038441c925bb""",2,2.637,0.0216,2018-08-14,15
"""038441c925bb""",3,2.6368,0.0213,2018-08-14,15
"""038441c925bb""",4,2.6368,0.0215,2018-08-14,15
…,…,…,…,…,…
"""0402a003dae9""",145,-59.696899,0.0601,2018-12-18,12
"""0402a003dae9""",146,-35.656601,0.0427,2018-12-18,12
"""0402a003dae9""",147,-21.582399,0.0309,2018-12-18,12


In [ ]:
print('Maximum steps per series in the current test data:')
display(test.group_by('series_id').agg(pl.col('step').max().alias('max_step')))

Maximum steps per series in the current test data:


series_id,max_step
str,u32
"""0402a003dae9""",149
"""038441c925bb""",149
"""03d92c9f6f8a""",149


In [ ]:
test

series_id,step,anglez,enmo,date,hour
str,u32,f32,f32,date,i8
"""038441c925bb""",0,2.6367,0.0217,2018-08-14,15
"""038441c925bb""",1,2.6368,0.0215,2018-08-14,15
"""038441c925bb""",2,2.637,0.0216,2018-08-14,15
"""038441c925bb""",3,2.6368,0.0213,2018-08-14,15
"""038441c925bb""",4,2.6368,0.0215,2018-08-14,15
…,…,…,…,…,…
"""0402a003dae9""",145,-59.696899,0.0601,2018-12-18,12
"""0402a003dae9""",146,-35.656601,0.0427,2018-12-18,12
"""0402a003dae9""",147,-21.582399,0.0309,2018-12-18,12


In [ ]:
# this is correct — submit the empty preds as-is
preds = predict(test, xgb).drop('date', axis=1)
preds.to_csv('submission.csv', index=False)

Processing users: 100%|██████████| 3/3 [00:00<00:00, 39.96it/s]


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report

all_series_ids = data_labeled.select("series_id").unique().to_series().to_list()
train_series_ids, holdout_series_ids = train_test_split(
    all_series_ids, test_size=0.1, random_state=SEED
)

train_data = data_labeled.filter(pl.col("series_id").is_in(train_series_ids))
holdout_data = data_labeled.filter(pl.col("series_id").is_in(holdout_series_ids))

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score

# ---- holdout split ----
all_series_ids = data_labeled.select("series_id").unique().to_series().to_list()
train_series_ids, holdout_series_ids = train_test_split(
    all_series_ids, test_size=0.1, random_state=SEED
)
train_data    = data_labeled.filter(pl.col("series_id").is_in(train_series_ids))
holdout_data  = data_labeled.filter(pl.col("series_id").is_in(holdout_series_ids))

# ---- training config ----
non_feat_cols = ['series_id', 'step', 'date', 'asleep']
batch_size = 1_000_000
rounds_per_batch = 50

label_counts = train_data.select(pl.col("asleep").value_counts()).unnest("asleep")
counts = dict(zip(label_counts["asleep"], label_counts["count"]))
scale_pos_weight = counts.get(0, 1) / max(counts.get(1, 1), 1)

params = {
    'tree_method': 'hist',
    'objective': 'binary:logistic',
    'eval_metric': 'logloss',
    'max_depth': 6,
    'eta': 0.1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'scale_pos_weight': scale_pos_weight,
    'seed': SEED,
    'nthread': -1,
}

n_batches = n_batches_for(train_data, batch_size)
booster = None

for i, features in tqdm(enumerate(batch_data(train_data, batch_size)),
                         total=n_batches, desc='Training batches'):
    X_batch = features.drop(non_feat_cols).to_numpy().astype('float32')
    y_batch = features.select("asleep").to_numpy().ravel().astype('int32')

    n = len(y_batch)
    cut = int(n * 0.9)
    dtrain = xgb_lib.DMatrix(X_batch[:cut], label=y_batch[:cut])
    dvalid = xgb_lib.DMatrix(X_batch[cut:], label=y_batch[cut:])

    booster = xgb_lib.train(
        params,
        dtrain,
        num_boost_round=rounds_per_batch,
        xgb_model=booster,
        evals=[(dvalid, 'valid')],
        early_stopping_rounds=10,
        verbose_eval=False,
    )

    del X_batch, y_batch, dtrain, dvalid, features
    gc.collect()

xgb = booster

# ---- event-level F1 on holdout (competition-style, ±30 min tolerance) ----
TOLERANCE = 12 * 30  # 360 steps = 30 minutes

holdout_preds_df = predict(holdout_data, xgb)

true_events = (
    events
    .filter(pl.col("series_id").is_in(holdout_series_ids))
    .drop_nulls(subset=["step"])
    .with_columns(pl.col("step").cast(pl.UInt32))
    .to_pandas()
)

results = []
for event_type in ["onset", "wakeup"]:
    true_ev = true_events[true_events["event"] == event_type][["series_id", "step"]].copy()
    pred_ev = holdout_preds_df[holdout_preds_df["event"] == event_type][["series_id", "step"]].copy()

    matched_true = set()
    matched_pred = set()

    for pred_idx, pred_row in pred_ev.iterrows():
        sid = pred_row["series_id"]
        pred_step = pred_row["step"]

        candidates = true_ev[
            (true_ev["series_id"] == sid) &
            (~true_ev.index.isin(matched_true)) &
            (abs(true_ev["step"] - pred_step) <= TOLERANCE)
        ]

        if not candidates.empty:
            closest_idx = (candidates["step"] - pred_step).abs().idxmin()
            matched_true.add(closest_idx)
            matched_pred.add(pred_idx)

    tp = len(matched_true)
    fp = len(pred_ev) - tp
    fn = len(true_ev) - tp

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    results.append({
        "event": event_type, "TP": tp, "FP": fp, "FN": fn,
        "precision": round(precision, 4),
        "recall":    round(recall, 4),
        "f1":        round(f1, 4),
    })
    print(f"\n--- {event_type.upper()} ---")
    print(f"  TP={tp}, FP={fp}, FN={fn}")
    print(f"  Precision: {precision:.4f}  Recall: {recall:.4f}  F1: {f1:.4f}")

results_df = pd.DataFrame(results)
macro_f1 = results_df["f1"].mean()
print(f"\n=== Macro F1 (onset + wakeup avg): {macro_f1:.4f} ===")
print(results_df.to_string(index=False))

Processing users: 100%|██████████| 28/28 [02:46<00:00,  5.96s/it]



--- ONSET ---
  TP=87, FP=410, FN=358
  Precision: 0.1751  Recall: 0.1955  F1: 0.1847

--- WAKEUP ---
  TP=129, FP=368, FN=316
  Precision: 0.2596  Recall: 0.2899  F1: 0.2739

=== Macro F1 (onset + wakeup avg): 0.2293 ===
 event  TP  FP  FN  precision  recall     f1
 onset  87 410 358     0.1751  0.1955 0.1847
wakeup 129 368 316     0.2596  0.2899 0.2739


In [ ]:
holdout_features = create_features(holdout_data)

X_ho = holdout_features.drop(non_feat_cols).to_numpy().astype('float32')
y_ho = holdout_features.select("asleep").to_numpy().ravel().astype('int32')

holdout_probs = xgb.predict(xgb_lib.DMatrix(X_ho))
holdout_preds = (holdout_probs >= 0.5).astype(int)

accuracy = (holdout_preds == y_ho).mean()
print(f"Row-level Accuracy: {accuracy:.4f}")

Row-level Accuracy: 0.9328
